# Auditoria das medidas DAX do CanaVis (Etapa 2)

**Notebook:** `notebooks/05_auditoria_medidas_dax.ipynb`
**Escopo desta rodada:** APENAS **auditar, reproduzir, diagnosticar, documentar e PROPOR**.
Nenhuma linha de `powerbi/modelo.tmdl` ou `powerbi/canavis.pbix` e alterada aqui. Todo ajuste de DAX
que eventualmente apareca entra como **proposta** ("sugestao, nao aplicada"); a decisao de aplicar e do time
depois de ler o relatorio `docs/entrega-etapa2/auditoria-medidas-dax.md`.

## Objetivo

A colega Alex pediu a revisao de cinco medidas, verificando se estao sendo calculadas corretamente num
dashboard cuja perspectiva e **"Paraiba vs outros estados"**:

1. **Produtividade** (`fato_producao`)
2. **Participacao regional** (`fato_producao`)
3. **Desvio vs selecao** (`fato_producao`)
4. **Desvio vs grupo** (`fato_producao`)
5. **Vendas etanol** — UF (`fato_vendas_etanol_uf`) e municipios validados (`fato_vendas_etanol_municipio`)

Para cada medida seguimos uma estrutura fixa de seis subpartes:

1. **Definicao atual** — DAX real colado de `powerbi/modelo.tmdl` e explicado termo a termo em pt-BR.
2. **Reproducao em pandas** em tres cenarios de contexto de filtro.
3. **Casos de teste numericos** com os numeros reais da gold (conferidos com `assert`).
4. **Diagnostico** rotulado `CORRETA` / `CORRETA-COM-RESSALVA` / `TEM PROBLEMA`, com evidencia numerica.
5. **Interpretacao e uso no Power BI**.
6. **Fundamentacao** na literatura brasileira/estatistica, com links.

## Ancoragem (decisoes e docs ja escritos)

- **ADR 0002** (`docs/decisoes/0002-fontes-publicas.md`): CONAB (`fato_industria`) so existe no nivel **UF**.
- **ADR 0006** (`docs/decisoes/0006-tiers-de-comparacao-por-clusterizacao.md`): `tier` so e definido nas 27 UFs;
  **Paraiba = `media escala`**.
- **ADR 0008** (`docs/decisoes/0008-tratamento-etapa2.md`): **zero != ausente**. `NaN` = "nao reportado",
  nunca coagido a zero; zero so quando a fonte afirma zero. As colunas `situacao_quantidade`/`situacao_area`
  classificam cada parcela (`presente`, `sem_producao`, `suprimido`).
- `docs/entrega-etapa2/modelo-dados.md` e `docs/entrega-etapa2/medidas-industriais.md`: regras das medidas,
  numeros de referencia e a validacao executada no Power BI Desktop.

> **Regra de ouro desta auditoria:** toda afirmacao numerica abaixo e reproduzida da gold real
> (`data/processed/*.parquet`). Se a gold mudar, os `assert` de sanidade quebram alto, de proposito.


In [1]:
# Bloco A.2 - imports, leitura da gold e asserts de sanidade.
# Executar a partir de notebooks/: a gold fica em ../data/processed.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt  # noqa: F401  (usado nas secoes visuais da FEAT seguinte)
from pathlib import Path

pd.set_option("display.float_format", lambda v: f"{v:,.10f}")

PROC = Path("../data/processed")

fato_producao = pd.read_parquet(PROC / "fato_producao.parquet")
fato_industria = pd.read_parquet(PROC / "fato_industria.parquet")
fato_vendas_uf = pd.read_parquet(PROC / "fato_vendas_etanol_uf.parquet")
fato_vendas_mun = pd.read_parquet(PROC / "fato_vendas_etanol_municipio.parquet")
dim_territorio = pd.read_parquet(PROC / "dim_territorio.parquet")

# ---- Asserts de sanidade: reproduzem os numeros-chave; falham alto se a gold mudar. ----

# fato_producao: 5.733 linhas x 17 colunas, tres niveis sobrepostos.
assert fato_producao.shape == (5733, 17), fato_producao.shape
assert fato_producao["nivel_territorial"].value_counts().to_dict() == {
    "municipio": 4683,
    "uf": 567,
    "microrregiao": 483,
}
# situacao_* classifica cada parcela (ADR 0008): presente / sem_producao / suprimido.
assert fato_producao["situacao_quantidade"].value_counts().to_dict() == {
    "sem_producao": 3036,
    "presente": 2671,
    "suprimido": 26,
}

# dim_territorio: 273 linhas; tier so nas 27 UFs; PB = uf:25, tier 'media escala'.
assert dim_territorio.shape[0] == 273
assert dim_territorio["nivel_territorial"].value_counts().to_dict() == {
    "municipio": 223,
    "uf": 27,
    "microrregiao": 23,
}
tier_por_nivel = (
    dim_territorio.groupby("nivel_territorial")["tier"].apply(lambda s: int(s.notna().sum())).to_dict()
)
assert tier_por_nivel == {"microrregiao": 0, "municipio": 0, "uf": 27}, tier_por_nivel
pb = dim_territorio.loc[dim_territorio["territorio_id"] == "uf:25"].iloc[0]
assert pb["nome_territorio"] == "Paraíba" and pb["tier"] == "media escala"

# CONAB (fato_industria): so nivel uf, 470 linhas, 24 UFs, 22 safras, sem coluna ano_referencia.
assert fato_industria.shape[0] == 470
assert fato_industria["territorio_id"].str.split(":").str[0].unique().tolist() == ["uf"]
assert fato_industria["territorio_id"].nunique() == 24
assert fato_industria["safra"].nunique() == 22
assert "ano_referencia" not in fato_industria.columns
assert int(fato_industria["ano_alinhamento"].max()) == 2026
# Perfil de ausencia/zeros (zero != ausente): base das secoes de cobertura da FEAT seguinte.
assert fato_industria["producao_acucar_mil_t"].isna().sum() == 7
assert (fato_industria["producao_acucar_mil_t"] == 0).sum() == 71
assert (fato_industria["producao_etanol_total_mil_l"] == 0).sum() == 6
assert (fato_industria["cana_mil_t"] == 0).sum() == 4
assert (fato_industria["atr_kg_t"] == 0).sum() == 6

# Vendas municipais: status_chave; so PB conferida tem territorio_id preenchido.
assert fato_vendas_mun["status_chave"].value_counts().to_dict() == {
    "nao_validada_fora_pb": 148244,
    "codigo_nome_conferidos_pb": 3581,
    "divergente_ou_sem_correspondencia": 1269,
}

print("Asserts de sanidade OK. Formas das tabelas:")
for nome, df in [
    ("fato_producao", fato_producao),
    ("fato_industria", fato_industria),
    ("fato_vendas_etanol_uf", fato_vendas_uf),
    ("fato_vendas_etanol_municipio", fato_vendas_mun),
    ("dim_territorio", dim_territorio),
]:
    print(f"  {nome:32s} {df.shape}")


Asserts de sanidade OK. Formas das tabelas:
  fato_producao                    (5733, 17)
  fato_industria                   (470, 17)
  fato_vendas_etanol_uf            (945, 7)
  fato_vendas_etanol_municipio     (153094, 11)
  dim_territorio                   (273, 13)


In [2]:
# Bloco A.3 - helpers puros que reproduzem o DAX das medidas em pandas.
# Cada funcao tem docstring citando o trecho de DAX que replica. Vivem no notebook
# e sao reutilizados pela secao/FEAT seguinte. Convencao: NaN = ausente, nunca zero.

NIVEL = "nivel_territorial"
Q = "quantidade_produzida_t"
A = "area_colhida_ha"


def nivel_unico(df: pd.DataFrame) -> bool:
    """Emula HASONEVALUE(dim_territorio[nivel_territorial]).

    Verdadeiro quando o contexto de filtro deixa exatamente UM nivel territorial
    visivel. No DAX de 'Produtividade': IF(HASONEVALUE(dim_territorio[nivel_territorial]) && ...).
    """
    return df[NIVEL].nunique(dropna=False) == 1


def tem_parcela_ausente(df: pd.DataFrame, colunas) -> bool:
    """Emula COUNTROWS(FILTER(fato, ISBLANK(col) || ...)) > 0.

    Verdadeiro se qualquer linha do contexto tem valor ausente (NaN) em alguma
    das colunas exigidas. No DAX, essa guarda invalida o agregado inteiro.
    """
    sub = df[list(colunas)]
    return bool(sub.isna().any(axis=1).any())


def produtividade(df: pd.DataFrame) -> float:
    """Emula a medida 'Produtividade':

    IF(HASONEVALUE(dim_territorio[nivel_territorial])
       && COUNTROWS(FILTER(fato_producao,
            ISBLANK(quantidade_produzida_t) || ISBLANK(area_colhida_ha))) = 0,
       DIVIDE(SUM(quantidade_produzida_t), SUM(area_colhida_ha)))

    Razao de agregados (media ponderada), NUNCA media das produtividades.
    Retorna NaN quando: nivel nao e unico, ha parcela ausente, ou DIVIDE protege
    denominador zero (area total 0). 'suprimido' aparece como NaN na gold, logo
    a guarda de ausencia ja o exclui do agregado.
    """
    if not nivel_unico(df):
        return np.nan
    if tem_parcela_ausente(df, (Q, A)):
        return np.nan
    soma_area = df[A].sum()
    if soma_area == 0:  # DIVIDE protege denominador zero -> BLANK
        return np.nan
    return df[Q].sum() / soma_area


def participacao(df_sel: pd.DataFrame, df_nivel: pd.DataFrame) -> float:
    """Emula a medida 'Participacao regional':

    numerador = SUM(q) do territorio selecionado (BLANK se houver parcela ausente);
    denominador = CALCULATE(SUM(q), ALL(dim_territorio), nivel_territorial = Nivel)
    (total do MESMO nivel, tambem BLANK se houver parcela ausente).
    Exige nivel unico na selecao (SELECTEDVALUE(nivel) nao-BLANK).

    df_sel  = fato ja filtrado pela selecao (ex.: PB, ano, fonte).
    df_nivel = fato filtrado pelo mesmo nivel e demais filtros, mas sobre ALL(territorio)
               (ex.: TODAS as UFs do ano) -> o denominador.
    """
    if df_sel[NIVEL].nunique(dropna=False) != 1:
        return np.nan  # SELECTEDVALUE(nivel) = BLANK
    if tem_parcela_ausente(df_sel, (Q,)):
        return np.nan
    if tem_parcela_ausente(df_nivel, (Q,)):
        return np.nan
    total = df_nivel[Q].sum()
    if total == 0:
        return np.nan
    return df_sel[Q].sum() / total


def desvio(atual: float, referencia: float) -> float:
    """Emula o padrao de desvio:

    IF(NOT ISBLANK(Atual) && NOT ISBLANK(Referencia), DIVIDE(Atual - Referencia, Referencia)).

    Retorna NaN se qualquer operando for NaN (ausencia nao vira zero) ou se a
    referencia for zero (DIVIDE protegido).
    """
    if pd.isna(atual) or pd.isna(referencia):
        return np.nan
    if referencia == 0:
        return np.nan
    return (atual - referencia) / referencia


def referencia_allselected(df_sel_nivel: pd.DataFrame) -> float:
    """Emula CALCULATE([Produtividade], ALLSELECTED(dim_territorio)) de 'Desvio vs selecao'.

    A referencia e a produtividade (razao de agregados) sobre TODOS os territorios
    da selecao visivel, no mesmo nivel. df_sel_nivel = fato das UFs selecionadas no ano.
    """
    return produtividade(df_sel_nivel)


def referencia_grupo(df_sel_nivel: pd.DataFrame, tier: str) -> float:
    """Emula CALCULATE([Produtividade], ALLSELECTED(dim_territorio),
    KEEPFILTERS(dim_territorio[tier] = Tier)) de 'Desvio vs grupo'.

    Restringe a referencia ao grupo (tier) do territorio atual. Como tier so e
    definido nas 27 UFs, df_sel_nivel precisa vir ja juntado ao tier da dim.
    Fora do nivel UF, tier = NaN -> referencia BLANK (NaN).
    """
    if tier is None or (isinstance(tier, float) and pd.isna(tier)) or pd.isna(tier):
        return np.nan
    grupo = df_sel_nivel.loc[df_sel_nivel["tier"] == tier]
    return produtividade(grupo)


def _ufs_do_nivel(df_fato: pd.DataFrame, ano: int) -> pd.DataFrame:
    """Atalho de teste: fato de producao no nivel uf, para um ano."""
    return df_fato[(df_fato[NIVEL] == "uf") & (df_fato["ano_referencia"] == ano)]


print("Helpers definidos:", [
    "nivel_unico", "tem_parcela_ausente", "produtividade", "participacao",
    "desvio", "referencia_allselected", "referencia_grupo",
])


Helpers definidos: ['nivel_unico', 'tem_parcela_ausente', 'produtividade', 'participacao', 'desvio', 'referencia_allselected', 'referencia_grupo']


## 1. Produtividade

### 1.1 Definicao atual (DAX real)

```dax
Produtividade =
IF(
    HASONEVALUE(dim_territorio[nivel_territorial])
        && COUNTROWS(
            FILTER(
                fato_producao,
                ISBLANK(fato_producao[quantidade_produzida_t])
                    || ISBLANK(fato_producao[area_colhida_ha])
            )
        ) = 0,
    DIVIDE(SUM(fato_producao[quantidade_produzida_t]), SUM(fato_producao[area_colhida_ha]))
)
```

**Leitura termo a termo (pt-BR):**

- `HASONEVALUE(dim_territorio[nivel_territorial])` — so calcula se o contexto deixa **um unico nivel**
  territorial visivel (uf, microrregiao OU municipio). Impede misturar niveis que sao **totais sobrepostos**.
- `COUNTROWS(FILTER(... ISBLANK(quantidade) || ISBLANK(area) ...)) = 0` — **guarda de ausencia**: se qualquer
  linha do contexto tem quantidade ou area **ausente** (`NaN`), o agregado inteiro fica indisponivel. E a
  materializacao do **ADR 0008** (ausencia nao vira zero). As parcelas `suprimido` entram como `NaN` na gold,
  logo sao justamente o que essa guarda barra.
- `DIVIDE(SUM(quantidade), SUM(area))` — **razao de agregados**: soma primeiro as quantidades, soma as areas, e
  so entao divide. E uma **media ponderada** pela area, nao a media das produtividades municipais. `DIVIDE`
  devolve BLANK (vazio) se a area somada for zero, sem erro de divisao.
- Unidade do resultado: **t/ha** (`formatString: 0.00`).


In [3]:
# 1.2 Reproducao em pandas - tres cenarios de contexto de filtro.
ANO = 2023

# (a) Paraiba, nivel UF unico.
pb_uf = fato_producao[
    (fato_producao[NIVEL] == "uf")
    & (fato_producao["ano_referencia"] == ANO)
    & (fato_producao["territorio_id"] == "uf:25")
]
prod_pb = produtividade(pb_uf)

# (b) Todas as UFs (contexto ALLSELECTED = nivel uf unico, varios territorios).
todas_uf = _ufs_do_nivel(fato_producao, ANO)
prod_todas_uf = produtividade(todas_uf)

# (c) Mistura UF + microrregiao + municipio (sem filtro de nivel).
mistura = fato_producao[fato_producao["ano_referencia"] == ANO]
prod_mistura = produtividade(mistura)

print(f"(a) PB UF 2023 .................. {prod_pb:,.10f} t/ha")
print(f"(b) Todas as UFs 2023 (ponderada) {prod_todas_uf:,.10f} t/ha")
print(f"    media SIMPLES das prod. UF ... {todas_uf['produtividade_t_ha'].mean():,.10f} t/ha  (NAO e o que a medida faz)")
print(f"(c) Mistura 3 niveis 2023 ....... {prod_mistura}  (NaN: HASONEVALUE barra nivel nao unico)")


(a) PB UF 2023 .................. 64.6923844485 t/ha
(b) Todas as UFs 2023 (ponderada) 77.9009038820 t/ha
    media SIMPLES das prod. UF ... 61.1388677983 t/ha  (NAO e o que a medida faz)
(c) Mistura 3 niveis 2023 ....... nan  (NaN: HASONEVALUE barra nivel nao unico)


In [4]:
# 1.3 Casos de teste numericos (numeros reais da gold).
ESPERADO_PROD_PB_2023 = 64.6923844485

assert abs(prod_pb - ESPERADO_PROD_PB_2023) < 1e-9, prod_pb
# A medida e ponderada (77,90), nao a media simples (~61,14) das produtividades por UF.
assert abs(prod_todas_uf - 77.9009038820) < 1e-9, prod_todas_uf
assert abs(todas_uf["produtividade_t_ha"].mean() - 61.1388677983) < 1e-9
# Mistura de niveis => NaN (guarda HASONEVALUE).
assert pd.isna(prod_mistura)
print("OK: produtividade PB 2023 =", f"{prod_pb:,.10f}", "t/ha")


OK: produtividade PB 2023 = 64.6923844485 t/ha


### 1.4 Diagnostico: **CORRETA**

- **Razao de agregados confirmada.** Para a PB em 2023: q = 7.066.802 t, area = 109.237 ha →
  7.066.802 / 109.237 = **64,6923844485 t/ha**, exatamente o valor de referencia de `modelo-dados.md`.
- **Ponderada, nao media de razoes.** Sobre todas as UFs de 2023, a razao de agregados da **77,90 t/ha**,
  enquanto a media simples das produtividades por UF daria **61,14 t/ha**. A medida usa a forma correta
  (ponderada pela area): UFs grandes pesam mais, como manda a metodologia de rendimento medio.
- **`suprimido` nao entra como zero.** As 26 parcelas `suprimido` da gold estao com quantidade/area `NaN`;
  a guarda `COUNTROWS(FILTER(... ISBLANK ...))=0` torna o agregado indisponivel sempre que uma delas esta no
  contexto, em vez de somar zero e subestimar a produtividade (ADR 0008).
- **`HASONEVALUE` protege contra mistura de niveis.** No cenario (c), sem filtro de nivel, a medida devolve
  BLANK em vez de um numero sem sentido somando municipio + microrregiao + UF.

**Ressalva de uso (nao e erro da formula):** a protecao depende de o visual deixar **um unico nivel** no
contexto. Num cartao sem o nivel fixado, a medida fica vazia — o que e seguro, mas pode confundir quem espera
um numero. Ver a secao de arquitetura (filtro de nivel) no relatorio.


### 1.5 Interpretacao e uso no Power BI

- **Onde aparece:** ranking de produtividade por UF e cartao de produtividade da PB (pagina "Validacao UF").
- **Como ler:** t de cana por hectare colhido no ano selecionado; quanto maior, mais eficiente a lavoura.
- **O que o filtro de nivel resolve:** fixar `nivel_territorial = uf` garante `HASONEVALUE` verdadeiro e um
  denominador coerente. Sem ele, a medida fica vazia (seguro) mas o visual parece "quebrado".
- **Armadilha:** nunca comparar este numero com uma media simples de produtividades municipais — sao
  grandezas diferentes (ponderada vs nao ponderada).

### 1.6 Fundamentacao

- Rendimento medio como **razao de agregados** (producao total / area total), nao media de razoes, e a
  convencao da [PAM-IBGE — Producao Agricola Municipal, conceitos e metodos](https://www.ibge.gov.br/estatisticas/economicas/agricultura-e-pecuaria/9117-producao-agricola-municipal-culturas-temporarias-e-permanentes.html).
  *Conteudo parafraseado para conformidade de licenciamento.*
- Contexto agronomico de rendimento da cana: [Embrapa Agroenergia](https://www.embrapa.br/agencia-de-informacao-tecnologica/tematicas/agroenergia/alcool).


## 2. Participacao regional

### 2.1 Definicao atual (DAX real)

```dax
Participação regional =
VAR Nivel = SELECTEDVALUE(dim_territorio[nivel_territorial])
VAR Quantidade =
    IF(COUNTROWS(FILTER(fato_producao, ISBLANK(fato_producao[quantidade_produzida_t]))) = 0,
       SUM(fato_producao[quantidade_produzida_t]))
VAR Total =
    CALCULATE(
        IF(COUNTROWS(FILTER(fato_producao, ISBLANK(fato_producao[quantidade_produzida_t]))) = 0,
           SUM(fato_producao[quantidade_produzida_t])),
        ALL(dim_territorio),
        dim_territorio[nivel_territorial] = Nivel
    )
RETURN IF(NOT ISBLANK(Nivel) && NOT ISBLANK(Quantidade), DIVIDE(Quantidade, Total))
```

**Leitura termo a termo (pt-BR):**

- `SELECTEDVALUE(dim_territorio[nivel_territorial])` — pega o **unico** nivel visivel; se houver mais de um,
  devolve BLANK e a medida nao calcula.
- `Quantidade` — producao do territorio selecionado, **com a mesma guarda de ausencia** (BLANK se alguma
  parcela do contexto tem quantidade `NaN`).
- `Total` com `CALCULATE(..., ALL(dim_territorio), dim_territorio[nivel_territorial] = Nivel)` — **remove** o
  filtro de territorio (`ALL`) mas **reimpoe o mesmo nivel**: o denominador e o total **daquele nivel**
  (ex.: soma de todas as UFs), preservando ano e fonte. Tambem exige completude (sem `NaN`).
- `DIVIDE(Quantidade, Total)` — razao protegida contra denominador zero.
- Unidade: **percentual** (`formatString: 0.00%`).


In [5]:
# 2.2 Reproducao em pandas - tres cenarios.
# (a) PB no total das UFs (denominador = todas as UFs do ano, mesmo nivel).
part_pb = participacao(pb_uf, todas_uf)

# (b) Contexto "todas as UFs" como selecao unica: cada UF sobre o total UF.
#     Para a propria soma de todas as UFs a participacao e 100%.
part_total_uf = participacao(todas_uf, todas_uf)

# (c) Mistura de niveis: numerador sem nivel unico -> BLANK (SELECTEDVALUE vira BLANK).
part_mistura = participacao(mistura, mistura)

print(f"(a) Participacao PB no total UF 2023 . {part_pb*100:,.10f}%")
print(f"(b) Soma de todas as UFs sobre total . {part_total_uf*100:,.4f}%  (100% por definicao)")
print(f"(c) Mistura de niveis ............... {part_mistura}  (NaN: nivel nao unico)")


(a) Participacao PB no total UF 2023 . 0.9036158274%
(b) Soma de todas as UFs sobre total . 100.0000%  (100% por definicao)
(c) Mistura de niveis ............... nan  (NaN: nivel nao unico)


In [6]:
# 2.3 Casos de teste numericos.
ESPERADO_PART_PB_2023 = 0.009036158274  # 0,9036158274%

assert abs(part_pb - ESPERADO_PART_PB_2023) < 1e-12, part_pb
assert abs(part_total_uf - 1.0) < 1e-12
assert pd.isna(part_mistura)
print("OK: participacao PB 2023 =", f"{part_pb*100:,.10f}%")


OK: participacao PB 2023 = 0.9036158274%


In [7]:
# 2.3b Evidencia da MISTURA DE NIVEIS: somar os tres niveis da PB passa de 100%.
# Cada nivel e um TOTAL SOBREPOSTO do mesmo fenomeno; somar duplica a producao da PB.
f23 = fato_producao[fato_producao["ano_referencia"] == ANO]
total_uf_2023 = f23.loc[f23[NIVEL] == "uf", Q].sum()
micro_pb_2023 = f23.loc[(f23[NIVEL] == "microrregiao") & f23["territorio_id"].str.startswith("microrregiao:25"), Q].sum()
mun_pb_2023 = f23.loc[(f23[NIVEL] == "municipio") & f23["territorio_id"].str.startswith("municipio:25"), Q].sum()
soma_tres_niveis = total_uf_2023 + micro_pb_2023 + mun_pb_2023
pct_sobre_total_uf = soma_tres_niveis / total_uf_2023 * 100

print(f"Total UF 2023 ............. {total_uf_2023:,.0f} t")
print(f"+ microrregioes PB ....... {micro_pb_2023:,.0f} t")
print(f"+ municipios PB .......... {mun_pb_2023:,.0f} t")
print(f"= soma CEGA dos 3 niveis . {soma_tres_niveis:,.0f} t = {pct_sobre_total_uf:,.1f}% do total UF verdadeiro")
assert abs(pct_sobre_total_uf - 101.8) < 0.05, pct_sobre_total_uf
print("-> A soma passa de 100% porque micro e municipio PB repetem producao ja contida no total UF.")


Total UF 2023 ............. 782,058,236 t
+ microrregioes PB ....... 7,066,802 t
+ municipios PB .......... 7,066,802 t
= soma CEGA dos 3 niveis . 796,191,840 t = 101.8% do total UF verdadeiro
-> A soma passa de 100% porque micro e municipio PB repetem producao ja contida no total UF.


### 2.4 Diagnostico: **CORRETA-COM-RESSALVA**

- **A formula esta correta** e bem protegida: PB em 2023 = **0,9036158274%** do total das 27 UFs (denominador
  completo, todas as UFs com `situacao_quantidade = presente`). A guarda de ausencia impede dividir por um
  total parcial, e `SELECTEDVALUE` zera o calculo se o nivel nao for unico.
- **Ressalva (mistura de niveis no visual).** Somar **cegamente** os tres niveis de 2023 (UF + microrregiao PB
  + municipio PB) da **796.191.840 t = 101,8%** do total verdadeiro das UFs — passa de 100% porque
  microrregiao e municipio sao **totais sobrepostos** que repetem a producao da PB ja contida no total UF.
  A medida **se protege** (SELECTEDVALUE vira BLANK sem nivel unico), mas um visual que **nao fixa o nivel**
  ainda engana: um ranking que mistura municipios e UFs perde sentido (percentuais incomparaveis, soma > 100%).
- **Conclusao:** o risco nao esta na expressao DAX, e na **montagem do visual**. O filtro de nivel territorial
  por pagina e hoje obrigatorio. A secao de arquitetura do relatorio propoe tornar isso estrutural.


### 2.5 Interpretacao e uso no Power BI

- **Onde aparece:** "Participacao por UF" — quanto cada UF representa na producao nacional do ano.
- **Como ler:** fatia percentual do territorio no total **do seu nivel**; so faz sentido comparar territorios
  do **mesmo** nivel.
- **O que o filtro de nivel resolve:** sem `nivel_territorial = uf` fixado, a medida fica vazia (protecao) ou,
  num visual mal montado que force agregacao, induz leitura de soma > 100%.
- **Armadilha:** nunca colocar municipio e UF no mesmo eixo de participacao.

### 2.6 Fundamentacao

- Participacao/concentracao regional e o campo do **Quociente Locacional (QL)** da economia regional,
  tratado por [IBGE](https://www.ibge.gov.br/) e [IPEA](https://www.ipea.gov.br/) *(links institucionais;
  pagina de metodologia especifica de QL a confirmar na escrita do relatorio)*. A medida aqui e a forma mais
  simples de participacao (parte sobre o todo do mesmo nivel), base do QL. *Conteudo parafraseado para
  conformidade de licenciamento.*


## 3. Desvio vs selecao

### 3.1 Definicao atual (DAX real)

```dax
Desvio vs seleção =
VAR Atual = [Produtividade]
VAR Referencia = CALCULATE([Produtividade], ALLSELECTED(dim_territorio))
RETURN IF(NOT ISBLANK(Atual) && NOT ISBLANK(Referencia), DIVIDE(Atual - Referencia, Referencia))
```

**Leitura termo a termo (pt-BR):**

- `Atual = [Produtividade]` — reaproveita a medida 1 (ja com todas as guardas de nivel unico e ausencia).
- `CALCULATE([Produtividade], ALLSELECTED(dim_territorio))` — recalcula a produtividade sobre **tudo que o
  usuario deixou selecionado**, ignorando o filtro de uma linha/territorio especifico mas respeitando os
  recortes externos (ano, fonte, segmentacoes). E a **referencia da selecao**. Como reusa `[Produtividade]`,
  tambem e uma razao de agregados (ponderada).
- `IF(NOT ISBLANK(Atual) && NOT ISBLANK(Referencia), ...)` — so calcula com **ambos** presentes: ausencia nao
  vira zero nem desvio de -100%.
- `DIVIDE(Atual - Referencia, Referencia)` — desvio percentual padrao, protegido contra referencia zero.
- Unidade: **percentual** (`formatString: 0.00%`).


In [8]:
# 3.2 Reproducao em pandas - tres cenarios.
# Referencia ALLSELECTED = produtividade ponderada de todas as UFs selecionadas.
ref_sel = referencia_allselected(todas_uf)

# (a) Desvio da PB vs a selecao (todas as UFs).
desvio_pb_sel = desvio(prod_pb, ref_sel)

# (b) Desvio da propria selecao vs ela mesma = 0.
desvio_sel_sel = desvio(prod_todas_uf, ref_sel)

# (c) Mistura de niveis: Atual = BLANK -> desvio BLANK.
desvio_mistura = desvio(produtividade(mistura), referencia_allselected(mistura))

print(f"Referencia ALLSELECTED (todas UF) . {ref_sel:,.10f} t/ha")
print(f"(a) Desvio PB vs selecao .......... {desvio_pb_sel*100:,.10f}%")
print(f"(b) Desvio selecao vs ela mesma ... {desvio_sel_sel*100:,.4f}%")
print(f"(c) Mistura de niveis ............. {desvio_mistura}  (NaN)")


Referencia ALLSELECTED (todas UF) . 77.9009038820 t/ha
(a) Desvio PB vs selecao .......... -16.9555406617%
(b) Desvio selecao vs ela mesma ... 0.0000%
(c) Mistura de niveis ............. nan  (NaN)


In [9]:
# 3.3 Casos de teste numericos.
ESPERADO_DESVIO_SEL_PB = -0.169555406617  # -16,9555406617%

assert abs(ref_sel - 77.9009038820) < 1e-9, ref_sel
assert abs(desvio_pb_sel - ESPERADO_DESVIO_SEL_PB) < 1e-10, desvio_pb_sel
assert abs(desvio_sel_sel) < 1e-12
assert pd.isna(desvio_mistura)
print("OK: desvio vs selecao PB 2023 =", f"{desvio_pb_sel*100:,.10f}%", "(ref", f"{ref_sel:,.10f} t/ha)")


OK: desvio vs selecao PB 2023 = -16.9555406617% (ref 77.9009038820 t/ha)


### 3.4 Diagnostico: **CORRETA-COM-RESSALVA**

- **Numericamente certa.** PB em 2023 fica **-16,9555406617%** abaixo da referencia ponderada das UFs
  (**77,9009038820 t/ha**), coerente com `modelo-dados.md`. A protecao `ISBLANK` evita o bug historico de
  desvio -100% quando a produtividade atual esta ausente (issue #11).
- **Ressalva 1 — `ALLSELECTED` sem nivel fixado.** A referencia depende do que esta **selecionado**. Se o
  visual nao fixa o nivel, `ALLSELECTED(dim_territorio)` pode trazer municipios, microrregioes e UFs juntos,
  produzindo uma referencia com mistura de niveis. Como `[Produtividade]` exige `HASONEVALUE`, na pratica a
  medida acaba BLANK nesse caso (protecao), mas o significado da "referencia" so e claro **com o nivel fixado**.
- **Ressalva 2 — referencia movel.** O numero muda conforme a selecao do usuario. E uma referencia **geral**
  da selecao, nao uma comparacao por pares; nao confundir com o desvio vs grupo (secao 4).


### 3.5 Interpretacao e uso no Power BI

- **Onde aparece:** comparacao geral — quanto a UF difere do conjunto que o usuario deixou selecionado.
- **Como ler:** sinal negativo = abaixo da referencia; positivo = acima. A referencia e a produtividade
  **ponderada** da selecao, nao a media simples.
- **O que o filtro de nivel resolve:** fixar `nivel_territorial = uf` garante que a referencia some "UF contra
  UFs", nao UF contra uma mistura.
- **Armadilha:** como a referencia e movel, documentar sempre **qual** selecao esta ativa ao citar o numero.

### 3.6 Fundamentacao

- Desvio percentual relativo a uma referencia e **estatistica descritiva basica** ((valor - referencia) /
  referencia); nao exige fonte externa. A escolha da referencia como **razao de agregados** da selecao herda a
  mesma fundamentacao da produtividade ([PAM-IBGE](https://www.ibge.gov.br/estatisticas/economicas/agricultura-e-pecuaria/9117-producao-agricola-municipal-culturas-temporarias-e-permanentes.html)).


## 4. Desvio vs grupo

### 4.1 Definicao atual (DAX real)

```dax
Desvio vs grupo =
VAR Tier = SELECTEDVALUE(dim_territorio[tier])
VAR Atual = [Produtividade]
VAR Referencia =
    CALCULATE([Produtividade], ALLSELECTED(dim_territorio),
              KEEPFILTERS(dim_territorio[tier] = Tier))
RETURN IF(NOT ISBLANK(Tier) && NOT ISBLANK(Atual) && NOT ISBLANK(Referencia),
          DIVIDE(Atual - Referencia, Referencia))
```

**Leitura termo a termo (pt-BR):**

- `SELECTEDVALUE(dim_territorio[tier])` — o grupo (tier) do territorio atual. `tier` so e definido nas **27
  UFs** (ADR 0006); microrregiao e municipio tem `tier = NaN`, logo `Tier` vira BLANK fora do nivel UF.
- `ALLSELECTED(dim_territorio)` — base da referencia (tudo selecionado), como na secao 3.
- `KEEPFILTERS(dim_territorio[tier] = Tier)` — **mantem** o filtro de grupo ao expandir: a referencia e a
  produtividade **apenas das UFs do mesmo tier** (ex.: `media escala`), nao de todas as UFs.
- `IF(NOT ISBLANK(Tier) && ...)` — exige tier presente **e** atual **e** referencia presentes; caso contrario
  BLANK. E a razao de a medida ficar vazia fora do nivel UF.
- Unidade: **percentual** (`formatString: 0.00%`).


In [10]:
# 4.2 Reproducao em pandas - tres cenarios.
# Juntar o tier da dim as UFs do ano (tier so existe no nivel uf).
tier_uf = dim_territorio.loc[dim_territorio[NIVEL] == "uf", ["territorio_id", "tier"]]
todas_uf_tier = todas_uf.merge(tier_uf, on="territorio_id", how="left")

tier_pb = dim_territorio.loc[dim_territorio["territorio_id"] == "uf:25", "tier"].iloc[0]

# (a) Desvio da PB vs seu grupo (media escala).
ref_grupo = referencia_grupo(todas_uf_tier, tier_pb)
desvio_pb_grupo = desvio(prod_pb, ref_grupo)

# (b) Quantas UFs no grupo 'media escala'.
n_media_escala = int((tier_uf["tier"] == "media escala").sum())

# (c) Fora do nivel UF (municipio): tier = NaN -> referencia e medida BLANK.
ref_grupo_municipio = referencia_grupo(todas_uf_tier, np.nan)

print(f"Tier da PB ................... {tier_pb!r}")
print(f"UFs no grupo 'media escala' .. {n_media_escala}")
print(f"Referencia do grupo .......... {ref_grupo:,.10f} t/ha")
print(f"(a) Desvio PB vs grupo ....... {desvio_pb_grupo*100:,.10f}%")
print(f"(c) tier NaN (fora de UF) .... {ref_grupo_municipio}  (NaN -> medida BLANK)")


Tier da PB ................... 'media escala'
UFs no grupo 'media escala' .. 14
Referencia do grupo .......... 62.5608197218 t/ha
(a) Desvio PB vs grupo ....... 3.4071879750%
(c) tier NaN (fora de UF) .... nan  (NaN -> medida BLANK)


In [11]:
# 4.3 Casos de teste numericos.
ESPERADO_DESVIO_GRUPO_PB = 0.034071879750  # +3,4071879750%

assert tier_pb == "media escala"
assert n_media_escala == 14, n_media_escala
assert abs(ref_grupo - 62.5608197218) < 1e-9, ref_grupo
assert abs(desvio_pb_grupo - ESPERADO_DESVIO_GRUPO_PB) < 1e-10, desvio_pb_grupo
assert pd.isna(ref_grupo_municipio)
print("OK: desvio vs grupo PB 2023 =", f"{desvio_pb_grupo*100:,.10f}%", "(ref grupo", f"{ref_grupo:,.10f} t/ha)")


OK: desvio vs grupo PB 2023 = 3.4071879750% (ref grupo 62.5608197218 t/ha)


### 4.4 Diagnostico: **CORRETA-COM-RESSALVA**

- **Numericamente certa no nivel UF.** PB (`media escala`, 14 UFs) fica **+3,4071879750%** acima da referencia
  ponderada do grupo (**62,5608197218 t/ha**). Contra a selecao inteira (secao 3) a PB estava **abaixo**;
  contra os pares de escala parecida, esta **acima** — exatamente o valor de comparar com quem e comparavel.
- **Ressalva estrutural — `tier` so existe nas UFs.** Como `tier = NaN` em microrregiao e municipio (ADR 0006),
  a medida retorna **BLANK fora do nivel UF**. Isso e **correto por design** (nao ha grupos definidos nesses
  niveis), mas num visual fora de UF a medida aparece vazia e pode ser lida como erro. A guarda
  `IF(NOT ISBLANK(Tier) && ...)` garante que nunca vira zero ou -100%.
- **Dependencia resolvida.** O `tier` ja esta materializado na gold para as 27 UFs (Task 4 / issue #8), entao
  a comparacao por pares e valida; antes disso a medida retornava vazio por falta da coluna.


### 4.5 Interpretacao e uso no Power BI

- **Onde aparece:** comparacao por pares — quanto a UF difere das UFs **do mesmo porte** (tier por clusterizacao,
  ADR 0006).
- **Como ler:** positivo = acima dos pares de escala semelhante; negativo = abaixo. Mais justo que o desvio vs
  selecao quando o objetivo e "comparar com quem e comparavel".
- **O que o filtro de nivel resolve:** so faz sentido no nivel UF; fora dele a medida fica (corretamente) vazia.
- **Armadilha:** nao interpretar o BLANK fora de UF como "zero" ou "sem diferenca".

### 4.6 Fundamentacao

- Comparar contra o grupo de pares (tier) reduz o vies de escala, no espirito da **analise de concentracao
  regional** ([IBGE](https://www.ibge.gov.br/), [IPEA](https://www.ipea.gov.br/)). A definicao dos tiers por
  clusterizacao esta no [ADR 0006](../docs/decisoes/0006-tiers-de-comparacao-por-clusterizacao.md). O calculo
  do desvio e estatistica descritiva (desvio percentual padrao). *Conteudo parafraseado para conformidade de
  licenciamento.*


## 5. Vendas etanol (UF e municipios validados)

### 5.1 Definicao atual (DAX real)

```dax
Vendas etanol UF (L) =
IF(SELECTEDVALUE(dim_territorio[nivel_territorial]) = "uf",
   SUM(fato_vendas_etanol_uf[vendas_etanol_hidratado_l]))

Vendas etanol municípios validados (L) =
IF(SELECTEDVALUE(dim_territorio[nivel_territorial]) = "municipio",
   SUM(fato_vendas_etanol_municipio[vendas_etanol_hidratado_l]))
```

**Leitura termo a termo (pt-BR):**

- `SELECTEDVALUE(dim_territorio[nivel_territorial]) = "uf"` / `= "municipio"` — cada medida so responde no **seu**
  nivel. Isso impede que as duas representacoes das mesmas vendas sejam somadas no mesmo visual.
- `SUM(...[vendas_etanol_hidratado_l])` — soma simples do volume de etanol hidratado, em **litros**.
- A medida municipal depende do **relacionamento** `fato_vendas_etanol_municipio[territorio_id]` →
  `dim_territorio`. So as linhas com `territorio_id` preenchido (PB conferida, `status_chave =
  codigo_nome_conferidos_pb`) se ligam a um municipio da dimensao; as demais ficam fora do visual municipal.
- Unidade: **litros** (`formatString: #,0`).

> **Nota (ADR 0008 / modelo-dados.md):** nao somar vendas UF com o detalhe municipal — sao duas
> representacoes das MESMAS vendas. O total por UF deve vir da fato de UF, que preserva valores cuja chave
> municipal nao foi validada.


In [12]:
# 5.2 Reproducao em pandas - tres cenarios.
# (a) Vendas UF da PB em 2023 (nivel uf).
vendas_uf_pb = fato_vendas_uf[
    (fato_vendas_uf["territorio_id"] == "uf:25") & (fato_vendas_uf["ano_referencia"] == ANO)
]["vendas_etanol_hidratado_l"].sum()

# (b) Soma dos municipios PB validados em 2023 (nivel municipio, chave conferida).
vm_pb = fato_vendas_mun[(fato_vendas_mun["uf"] == "PB") & (fato_vendas_mun["ano_referencia"] == ANO)]
vendas_mun_pb = vm_pb["vendas_etanol_hidratado_l"].sum()
n_mun_pb = vm_pb["codigo_municipio_origem"].nunique()
status_pb = vm_pb["status_chave"].value_counts().to_dict()

# (c) Linhas fora da PB nao tem territorio_id -> nao entram no visual municipal validado.
sem_territorio = fato_vendas_mun["territorio_id"].isna().sum()
com_territorio = fato_vendas_mun["territorio_id"].notna().sum()

print(f"(a) Vendas UF PB 2023 ............ {vendas_uf_pb:,.0f} L")
print(f"(b) Soma municipios PB 2023 ..... {vendas_mun_pb:,.0f} L  ({n_mun_pb} municipios)")
print(f"    status_chave dos municipios PB: {status_pb}")
print(f"(c) linhas municipais SEM territorio_id (fora da PB): {sem_territorio:,} | COM: {com_territorio:,}")


(a) Vendas UF PB 2023 ............ 126,411,400 L
(b) Soma municipios PB 2023 ..... 126,411,400 L  (159 municipios)
    status_chave dos municipios PB: {'codigo_nome_conferidos_pb': 159}
(c) linhas municipais SEM territorio_id (fora da PB): 149,513 | COM: 3,581


In [13]:
# 5.3 Casos de teste numericos.
ESPERADO_VENDAS_PB_2023 = 126_411_400

assert vendas_uf_pb == ESPERADO_VENDAS_PB_2023, vendas_uf_pb
# A soma municipal validada coincide com o total UF nesse ano: todos os 159 municipios PB
# de 2023 tem status_chave = codigo_nome_conferidos_pb (todos conferidos).
assert vendas_mun_pb == ESPERADO_VENDAS_PB_2023, vendas_mun_pb
assert n_mun_pb == 159, n_mun_pb
assert status_pb == {"codigo_nome_conferidos_pb": 159}, status_pb
print("OK: vendas UF PB 2023 =", f"{vendas_uf_pb:,.0f} L", "| soma municipios PB =", f"{vendas_mun_pb:,.0f} L")


OK: vendas UF PB 2023 = 126,411,400 L | soma municipios PB = 126,411,400 L


### 5.4 Diagnostico: **CORRETA-COM-RESSALVA**

- **Numeros batem.** Vendas UF da PB em 2023 = **126.411.400 L**; a soma dos **159 municipios PB validados**
  do mesmo ano da **exatamente 126.411.400 L**. Coincidem porque, em 2023, **todos** os municipios PB tem
  `status_chave = codigo_nome_conferidos_pb` — a validacao esta completa, entao detalhe municipal e total UF
  conferem.
- **Separacao por nivel correta.** Cada medida so responde no seu nivel (`= "uf"` / `= "municipio"`), evitando
  somar as duas representacoes das mesmas vendas num unico visual (regra do `modelo-dados.md`).
- **Ressalva — cobertura do detalhe municipal.** So linhas com `territorio_id` preenchido (PB conferida) se
  ligam a um municipio da dimensao. Das 153.094 linhas municipais, a imensa maioria (`nao_validada_fora_pb`,
  `divergente_ou_sem_correspondencia`) fica **fora** do visual municipal validado. Isso e **intencional** (so
  a PB foi conferida codigo+nome), mas significa que "Vendas etanol municipios validados" **nao e um total
  nacional** — e so a PB. Fora da PB, usar sempre a medida de **UF**, que preserva o valor mesmo sem chave
  municipal validada.


### 5.5 Interpretacao e uso no Power BI

- **Onde aparece:** consumo comercial de etanol hidratado por UF/ano (medida UF) e por municipio PB validado
  (medida municipal).
- **Como ler:** volume em **litros**. Para total por UF use a medida de UF; o detalhe municipal so e confiavel
  na PB.
- **O que o filtro de nivel resolve:** cada medida ja se auto-restringe ao seu nivel via `SELECTEDVALUE`, entao
  nao ha risco de dupla contagem mesmo sem filtro de pagina; ainda assim, fixar o nivel deixa o visual previsivel.
- **Armadilha:** nao ler "Vendas etanol municipios validados" como total do Brasil — e so a PB conferida.
  Atencao tambem ao `escopo_vendas`: vendas ate 2006 incluem consumo proprio na fonte (ADR 0008).

### 5.6 Fundamentacao

- As vendas vem da serie da ANP. A distincao entre o agregado por UF e o detalhe municipal (com validacao de
  chave codigo+nome) segue o principio **zero != ausente / chave nao validada != municipio conhecido** do
  [ADR 0008](../docs/decisoes/0008-tratamento-etapa2.md). Soma simples de volumes em litros e agregacao direta
  (sem razao), entao a fundamentacao estatistica e a de totalizacao de contagem/volume. *Conteudo parafraseado
  para conformidade de licenciamento.*


## Encerramento (FEAT-001)

Esta rodada estabeleceu o notebook (Bloco A: setup + helpers) e auditou as cinco medidas (Bloco B), cada uma
com DAX real, reproducao em tres cenarios, casos de teste numericos conferidos, diagnostico rotulado,
interpretacao de uso e fundamentacao. Resumo dos vereditos:

| Medida | Veredito | Evidencia-chave |
|---|---|---|
| Produtividade | CORRETA | 64,6923844485 t/ha (ponderada 77,90 vs media simples 61,14) |
| Participacao regional | CORRETA-COM-RESSALVA | PB 0,9036158274%; soma cega dos 3 niveis = 101,8% |
| Desvio vs selecao | CORRETA-COM-RESSALVA | -16,9555406617% (ref ALLSELECTED 77,9009038820) |
| Desvio vs grupo | CORRETA-COM-RESSALVA | +3,4071879750% (ref grupo `media escala` 62,5608197218) |
| Vendas etanol (UF/municipios) | CORRETA-COM-RESSALVA | 126.411.400 L (UF = soma municipal PB validada) |

As secoes de volume/mix (ATR-equivalente, cobertura CONAB) e a secao de arquitetura do filtro de nivel ficam
para a FEAT seguinte, que reutiliza os helpers definidos no Bloco A.3. **Nenhum ajuste foi aplicado ao modelo
Power BI**; propostas de DAX, se houver, vivem no relatorio como "sugestao, nao aplicada".
